# **PicassoPy Workshop --> Test case: cpv 2024-08-21**
---


- Folder for data `test_case_data` -> download separately
- Folder for configs `test_case_config`


## Imports

In [1]:
import os, sys
import argparse
import datetime
import logging
from pathlib import Path
import numpy as np

sys.path.append('../')
import ppcpy
import ppcpy.io.loadConfigs as loadConfigs
import ppcpy.io.readPollyRawData as readPollyRawData
import ppcpy.interface.picassoProc as picassoProc
import ppcpy.misc.helper as helper
import ppcpy.misc.startscreen as startscreen
from ppcpy.io.write2nc import write_channelwise_2_nc_file, write2nc_file, write_profile2nc_file

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.tab20.colors)

## Defining Script Inputs

- The parameters `args.device`, `args.timestamp`, `args.picasso_config_file`, `args.level0_file_to_process`, need to be manually specified per case.
- The parameter `DATABASE_PATH` is the path to the database used for storing the retrieved calibration constants

In [2]:
## For purpose of the notebook mimic the argparse interface
from types import SimpleNamespace
args = SimpleNamespace()

## The used device and time of measurement
args.device = 'pollyxt_cpv'
args.timestamp = '20240821'
dt = datetime.datetime.strptime(args.timestamp, "%Y%m%d")

## The used config file
args.picasso_config_file = "test_case_config/pollynet_processing_chain_config_test.json"

## The data file to use
args.level0_file_to_process = f"test_case_data/{dt:%Y_%m_%d_%a}_CPV_00_00_01.nc"

In [3]:
startscreen.startscreen()

      ____  _                            ____           ___ ____ 
     / __ \(_)________ _______________  / __ \__  __   <  // __ \
    / /_/ / / ___/ __ `/ ___/ ___/ __ \/ /_/ / / / /   / // / / /
   / ____/ / /__/ /_/ (__  |__  ) /_/ / ____/ /_/ /   / // /_/ / 
  /_/   /_/\___/\__,_/____/____/\____/_/    \__, /   /_(_)____/  
                                           /____/                


## Load Data and Config-files

Loads data and information from the config files into the following three dictionaries:
- `picasso_config_dict`: Paths and other information stored in the picasso config file
- `polly_config_dict`: Configuration variables from polly config and polly default files
- `rawdata_dict`: Measurement data and information extracted from the level0 file

In [4]:
## Path to default Picasso config file
picasso_default_config_file = Path(
    helper.detect_path_type(Path.cwd().parent), 'ppcpy', 'config', 'pollynet_processing_chain_config.json')

## Load Picasso config file
picasso_config_dict = loadConfigs.loadPicassoConfig(args.picasso_config_file, picasso_default_config_file)

## load polly config file
polly_config_array = loadConfigs.readPollyNetConfigLinkTable(
    picasso_config_dict['pollynet_config_link_file'], timestamp=args.timestamp, device=args.device
)
polly_config_dict = loadConfigs.getPollyConfigfromArray(
    polly_config_array, picasso_config_dict
)

## Load level0-data file
rawfile_fullname = args.level0_file_to_process
rawfile = helper.detect_path_type(rawfile_fullname)
rawdata_dict = readPollyRawData.readPollyRawData(rawfile)

2026-09-15 19:56:54,590 - INFO - picasso_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\pollynet_processing_chain_config.json
2026-09-15 19:56:54,592 - INFO - picasso_config_file: test_case_config/pollynet_processing_chain_config_test.json
2026-09-15 19:56:54,594 - INFO - pollynet_config_link_file: test_case_config/pollynet_processing_chain_config_links.xlsx
2026-09-15 19:56:54,931 - INFO - polly_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\polly_global_config.json
2026-09-15 19:56:54,934 - INFO - polly_config_file: test_case_config\pollyxt_cpv_config_20230927.json
2026-09-15 19:56:54,936 - INFO - keys default/template file, but not in specific file {'depolCaliConst355', 'logbookPath', 'zLim_VolDepol_355', 'depolCaliConst532', 'depol_cal_time_fixed_p_end', 'ice_thres_par_depol', 'radiosondeFolder', 'clear_thres_par_beta_1064', 'bgCorRangeIndxLow', 'volDepolerror532', 'unspheroid_thres_par_depol', 'logbookFileName', 'is460nm', 'flagUseMa

## Initialize PicassoProc object

PicassoProc is the main object in PicassoPy, and is responsible for running all processes included and storing the data.

In [5]:
## Initialize PicassoProc
data_cube = picassoProc.PicassoProc(rawdata_dict, polly_config_dict, picasso_config_dict)

In [6]:
## reset date if date in filename differs date within nc-file 
data_cube.reset_date_infile()

## checking for correct measurement shots
data_cube.check_for_correct_mshots()

## setting channelTags
data_cube.setChannelTags()

## check for correct date in nc-file
data_cube.reset_date_infile()

2026-09-15 19:56:57,391 - INFO - date consistency-check... 
2026-09-15 19:56:57,392 - INFO - ... date in nc-file equals date of filename
2026-09-15 19:56:57,394 - WARNING - Removed 'none' tag from channel list. Indices removed: [15]
2026-09-15 19:56:57,396 - INFO - ChannelTags: ['FR-total-355 nm', 'FR-cross-355 nm', 'FR-387 nm', 'FR-407 nm', 'FR-total-532 nm', 'FR-cross-532 nm', 'FR-607 nm', 'FR-total-1064 nm', 'NR-total-532 nm', 'NR-607 nm', 'NR-total-355 nm', 'NR-387 nm', 'DFOV', '1058', '1064s']
2026-09-15 19:56:57,398 - INFO - date consistency-check... 
2026-09-15 19:56:57,399 - INFO - ... date in nc-file equals date of filename


## Preprocessing & Saturation Detection

The preprocessing includes the following processes:
- Deadtime correction
- Background correction
- SNR claculations
- Flagging of data
- Range correction

In [7]:
## Perform preprocessing, this includes Dead-time correction, Background correction, and Range correction
data_cube.preprocessing(collect_debug=True)

2026-09-15 19:56:57,406 - INFO - Preprocessing ...
2026-09-15 19:56:57,408 - INFO - ... time conversion
2026-09-15 19:56:57,415 - WARNING - ... mShots not constant min 0 max 2999
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\preprocess\pollyPreprocess.py:507: RuntimeWarning: invalid value encountered in divide
  PCR = (c * signal)/(2 * hRes * mShots[:, np.newaxis, :])
2026-09-15 19:56:58,486 - INFO - ... calculate dead-time corrected signal
2026-09-15 19:56:58,487 - INFO - ... Deadtime-correction (Mode: 1)
2026-09-15 19:57:10,660 - INFO - ... calculate background corrected signal
2026-09-15 19:57:10,662 - INFO - ... removing background from signal
2026-09-15 19:57:13,136 - INFO - ... height bin calculations
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\preprocess\pollyPreprocess.py:378: UserWarning: no explicit representation of timezones available for np.datetime64
  data_dict['time64'] = np.array([np.datetime64(t) for t in mTime_obj])
2026-09-15 19:57:13,434 - INFO - ... 

In [8]:
## Save high resolution signal-to-noise ratio, background, and range corrected signal
write_channelwise_2_nc_file(data_cube, prod_ls=['SNR', 'BG', 'RCS'])

2026-09-15 19:57:19,451 - INFO - saving product: SNR
2026-09-15 19:57:19,632 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_SNR.nc
2026-09-15 19:57:21,939 - INFO - saving product: BG
2026-09-15 19:57:22,055 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_BG.nc
2026-09-15 19:57:22,064 - INFO - saving product: RCS
2026-09-15 19:57:22,175 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_RCS.nc


In [9]:
## Display available channels
data_cube.channel_dict

{0: 'FR-total-355 nm',
 1: 'FR-cross-355 nm',
 2: 'FR-387 nm',
 3: 'FR-407 nm',
 4: 'FR-total-532 nm',
 5: 'FR-cross-532 nm',
 6: 'FR-607 nm',
 7: 'FR-total-1064 nm',
 8: 'NR-total-532 nm',
 9: 'NR-607 nm',
 10: 'NR-total-355 nm',
 11: 'NR-387 nm',
 12: 'DFOV',
 13: '1058',
 14: '1064s'}

In [10]:
## Detect and flag saturated signal
data_cube.SaturationDetect()

2026-09-15 19:57:25,451 - INFO - Saturation detection ...
2026-09-15 19:57:25,452 - INFO - Saturation detection


## Depol Calibration


- Depol. calibration constants (DC) are retrieved at each depol. calibration period included in the data.
- If enabled, DCs from the time period [24h before the measurement, 24 after the measurement] are loaded from a dedicated database.
- The optimal DCs. ie. the ones used for processing, are chosen from the set of retrieved and loaded WVCs based on their respective uncertainty.
- If no DCs are found, default values will be used.

In [11]:
## Delta 90 polarization calibration
data_cube.polarizationCaliD90()

2026-09-15 19:57:37,677 - INFO - Starting loadGHK
2026-09-15 19:57:37,679 - INFO - Using GHK from config file
2026-09-15 19:57:37,680 - INFO - G: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.], H: [ 0.02041 -0.998    1.       1.      -0.01477 -0.9987   1.      -0.02439
  1.       1.       1.       1.       1.       1.      -0.996  ], K: [0.97859 1.      1.      1.      0.99343 1.      1.      0.9947  1.
 1.      1.      1.      1.      1.      1.     ]
2026-09-15 19:57:37,681 - INFO - Performing Delta-90° Depol calibration ...
2026-09-15 19:57:37,682 - INFO - Channels: 355 total FR | 355 cross FR
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:427: RuntimeWarning: divide by zero encountered in divide
  dplus = smooth_signal(sig_x_p, smooth_win) / smooth_signal(sig_t_p, smooth_win)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:427: RuntimeWarning: invalid value encountered in divide
  dplus = smooth_signal(sig_x_p, smooth

In [12]:
## Display depolarization calibration constants
data_cube.etaused

{'355_FR': {'eta': np.float64(47.55786879900888),
  'eta_std': np.float64(1.0323527156051746),
  'method': np.str_('D90')},
 '532_FR': {'eta': np.float64(12.691980299458336),
  'eta_std': np.float64(0.3597337223059366),
  'method': np.str_('D90')},
 '1064_FR': {'eta': np.float64(0.18186339128701293),
  'eta_std': np.float64(0.009052282683450666),
  'method': np.str_('D90')}}

## Cloud Screening

Three modes of cloud Screening are currently implemented:

0. No cloud screening. Return cloud free for all timestamps
1. Cloud screen with Maximum Gradiant Signal (MSG) algorithm
2. Cloud screen with Zhao's algorithm

Clouds are screened per timestamp (30s). After the screening the data is splitt up into cloud free segments and aggregated.

In [13]:
## Apply cloud screening
data_cube.cloudScreen()

2026-09-15 19:57:38,204 - INFO - Cloud screening ...
2026-09-15 19:57:38,205 - INFO - cloud screen mode 1: MSG method.
2026-09-15 19:57:38,493 - INFO - Skipping cloud screening for timestamp 1502.
2026-09-15 19:57:38,860 - INFO - Skipping cloud screening for timestamp 1502.


In [14]:
## Segmentate cloud free groups
data_cube.cloudFreeSeg()

2026-09-15 19:57:39,002 - INFO - Segment cloud free groups ...
2026-09-15 19:57:39,003 - INFO - intNProfiles: 120, minIntNProfiles: 30


In [15]:
## Display cloud free groups
data_cube.clFreeGrps

array([[  0, 113],
       [144, 244]])

In [16]:
## Aggregate background, background corrected signal, and range corrected signal
data_cube.aggregate_profiles()

2026-09-15 19:57:39,029 - INFO - Aggregating variable: sigBGCor ...
2026-09-15 19:57:39,068 - INFO - Aggregating variable: BG ...
2026-09-15 19:57:39,070 - INFO - Aggregating variable: RCS ...
2026-09-15 19:57:39,110 - INFO - Aggregating variable: mShots ...
2026-09-15 19:57:39,111 - INFO - Aggregating variable: mask387Off ...
2026-09-15 19:57:39,112 - INFO - Aggregating variable: mask607Off ...
2026-09-15 19:57:39,113 - INFO - Aggregating variable: mask407Off ...


## Molecular Profiles

The molecular profiles are calculated from cloudNet ECMWF model data. Gdas1 data is not supported in PicassoPy!

In [17]:
## QuickFix for loadMeteo bug:
METEO_DATA_DIR_PATH = "test_case_data"
data_cube.polly_config_dict['meteorDataSource'] = 'nc_cloudnet'
data_cube.polly_config_dict['meteo_folder'] = METEO_DATA_DIR_PATH
data_cube.polly_config_dict['meteo_file'] = r"[\\/]{0:%Y%m%d}_.*\.nc"

## Load meteorological data
data_cube.loadMeteo()

## Calculate molecular profiles
data_cube.calcMolecular()

2026-09-15 19:57:39,121 - INFO - Loading meteorological data ...
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\io\readMeteo.py:263: FutureWarning: In a future version, xarray will not decode the variable 'forecast_time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.load_dataset(filename)
2026-09-15 19:57:39,219 - INFO - Performing model height correction.
2026-09-15 19:57:39,221 - INFO - Uncorrected model height[:,0]:
[10.466079  10.4652    10.460253  10.455192  10.449221  10.447984
 10.453651  10.460172  10.464973  10.472917  10.479133  10.48550

## Water Vapor calibration

- Water vapor calibration constants (WVC) are retrieved at each cloud free period.
- If enabled, WVCs from the time period [24h before the measurement, 24 after the measurement] are loaded from a dedicated database.
- The optimal WVC. ie. the one used for processing, is chosen from the set of retrieved and loaded WVCs based on their respective uncertainty.
- If no WVCs are found, default values will be used.

In [18]:
## Water Vapor Calibration
data_cube.watervaporCali()

2026-09-15 19:57:39,300 - INFO - Performing Water Vapor calibration ...
2026-09-15 19:57:39,301 - INFO - WVC retrieval method: model data
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\watervapor.py:164: RuntimeWarning: Mean of empty slice
  sigBGCor_387 = np.nanmean(sig387, axis=0)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\watervapor.py:165: RuntimeWarning: Mean of empty slice
  sigBGCor_407 = np.nanmean(sig407, axis=0)
2026-09-15 19:57:39,327 - INFO - cldFreGrp 0:
  WVC  (profile)       = 6.04 +/- 5.55
  WVC  (regression)    = 5.98 +/- 3.59  (R2=0.80)
2026-09-15 19:57:39,343 - INFO - cldFreGrp 1:
  WVC  (profile)       = 6.97 +/- 4.19
  WVC  (regression)    = 6.23 +/- 3.35  (R2=0.83)
2026-09-15 19:57:39,344 - INFO - Loading Water Vapor calibration constants from database: pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:57:39,349 - INFO - Loaded 2 lines from table 'wv_calibration_constant' with 'Model_Profile_Method'.
2026-09-15 19:57:39,

In [19]:
# Display water vapor calibration constant
data_cube.WVCused

{'407_FR': {'WVC': np.float64(6.229755359510672),
  'WVCStd': np.float64(3.345016754556772),
  'method': np.str_('model_regression')}}

## Rayleigh-Fit

- Douglas-Peucker algorithm is used to segment the signal into potential reference heights
- The reference height with the best fit to the molecular backscatter per channel is chosen
- Currently only done for FR-channels. NR reference heights are read from the config variables `refH_NR_{wavelength}`

In [20]:
## Rayleigh-fit procedure --> produces the reference heights
data_cube.rayleighFit()

2026-09-15 19:57:39,371 - INFO - Start Rayleigh Fit
2026-09-15 19:57:39,371 - WARNING - Potential for differences to matlab code due to numerical issues (subtraction of two small values)
2026-09-15 19:57:39,372 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-09-15 19:57:39,374 - WARNING - at 10km height this is a difference of about 4 indices
2026-09-15 19:57:39,374 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-09-15 19:57:39,376 - INFO - Channel: 532 total FR.
2026-09-15 19:57:39,380 - WARNING - Warning: Odd smoothinglengs not allowd, will preform smoothing with length win - 1.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\rayleighfit.py:574: RuntimeWarning: divide by zero encountered in divide
  std_aer_norm = sig_aer_norm / np.sqrt(pc + bg)
2026-09-15 19:57:39,439 - INFO - Channel: 355 total FR.
2026-09-15 19:57:39,444 - WARNING - Warning: Odd smoothingl

In [21]:
## Display reference heights for a given cloud free group
grpIdx = 0
print(f"""Reference heights in meters for cloud free period {grpIdx} {data_cube.retrievals_highres['time64'][data_cube.clFreeGrps[grpIdx]]}:
355 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_FR"]['refHeight'], 0)}
532 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_FR"]['refHeight'], 0)}
1064 total FR: {np.round(data_cube.retrievals_profile['refH'][grpIdx]["1064_total_FR"]['refHeight'], 0)}
355 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_NR"]['refHeight'], 0)}
532 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_NR"]['refHeight'], 0)}""")

Reference heights in meters for cloud free period 0 ['2024-08-21T00:00:00.000000' '2024-08-21T00:56:30.000000']:
355 total FR:  [ 6877. 12849.]
532 total FR:  [15768. 17935.]
1064 total FR: [13102. 13512.]
355 total NR:  [3005. 3995.]
532 total NR:  [3005. 3995.]


## GHK-Transmission Correction

In [22]:
## Molecular polarization calibration 
data_cube.polarizationCaliMol()

2026-09-15 19:57:39,819 - WARNING - 'flagMolDepolCali' set to False


In [23]:
## Apply GHK-transmission correction
data_cube.transCor()

2026-09-15 19:57:39,835 - INFO - GHK Transmission correction ...
2026-09-15 19:57:40,027 - INFO - Channel: 355 total FR | 355 cross FR
2026-09-15 19:57:40,184 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.55786879900888 error [ 0.00016  0.01567 -0.00958] Window 1 
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:195: RuntimeWarning: divide by zero encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:195: RuntimeWarning: invalid value encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:205: RuntimeWarning: invalid value encountered in divide
  vol_depol = (sig_ratio / eta * (Gt + Ht) - (Gr + Hr)) / ((Gr - Hr) - sig_ratio / eta * (Gt - Ht))
2026-09-15 19:57:40,599 - INFO - Channel: 532 total FR | 532 cross FR
2026-09-15 19:57:40,797 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.691980299458336 erro

In [24]:
## Aggregate GHK-transmission corrected profiles
data_cube.aggregate_profiles(var=['sigTCor', 'BGTCor'])

2026-09-15 19:57:41,900 - INFO - Aggregating variable: sigTCor ...
2026-09-15 19:57:41,963 - INFO - Aggregating variable: BGTCor ...


## Klett and Raman retrieval

Produces the following profiles per channel:
- Klett: Aerosol Backscatter and Extinction
- Raman: Aerosol Backscatter, Aerosol Extinction, and Lidar Ratio

If `nr=True`, perform the retrievals for FR and NR channels. Otherwise only FR.

In [25]:
## Klett retrieval for GHK-transmission corrected profiles
data_cube.retrievalKlett(nr=True)

2026-09-15 19:57:41,972 - INFO - Klett retrieval for FR & NR GHK-transmission corrected signal ...
2026-09-15 19:57:41,973 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-09-15 19:57:41,974 - WARNING - at 10km height this is a difference of about 4 indices
2026-09-15 19:57:41,976 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-09-15 19:57:41,977 - INFO - Channel: 532, total, FR klett.
2026-09-15 19:57:42,011 - INFO - Channel: 355, total, FR klett.
2026-09-15 19:57:42,041 - INFO - Channel: 1064, total, FR klett.
2026-09-15 19:57:42,069 - INFO - Channel: 532, total, NR klett.
2026-09-15 19:57:42,098 - INFO - Channel: 355, total, NR klett.
2026-09-15 19:57:42,126 - INFO - Cloud free segment 1, Time: 2024-08-21T01:12:00.000000 - 2024-08-21T02:02:00.000000.
2026-09-15 19:57:42,127 - INFO - Channel: 532, total, FR klett.
2026-09-15 19:57:42,153 - INFO - Channel: 355, total, FR klett

In [26]:
## Raman retrieval for GHK-transmission corrected profiles
data_cube.retrievalRaman(nr=True)

2026-09-15 19:57:42,249 - INFO - Raman retrieval for FR & NR GHK-transmission corrected signal ...
2026-09-15 19:57:42,250 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-09-15 19:57:42,251 - WARNING - at 10km height this is a difference of about 4 indices
2026-09-15 19:57:42,251 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-09-15 19:57:42,252 - INFO - Channels: 355, total, FR | 387, total, FR raman.
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
2026-09-15 19:57:42,402 - INFO - Filling aerExt below overlap with 0.00010661377416860892 for calculating the backscatter
2026-09-15 19:57:42,920 - INFO - Channels: 532, total, FR | 607, total, FR raman.
2026-09-15 19:57:43,065 - INFO - Filling aerExt below

## Overlap Correction

Two methods are available for calculating the Overlap Function:

1. FRNR method
2. Raman method

And four methods (currently only 3 implemented) are available for applying the Overlap Correction:

0. no overlap correction
1. overlap correction with using the default overlap function (read function from file)
2. overlap correction with using the calculated overlap function
3. overlap correction with gluing near-range and far-range signal -> Not implemented yet!


In [27]:
## Calculate overlap function
data_cube.overlapCalc()

## Fix spike in lower bins
data_cube.overlapFixLowestBins()

## Apply overlap correction
data_cube.overlapCor()

2026-09-15 19:57:48,505 - INFO - calculating overlap functions ...
2026-09-15 19:57:48,506 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-09-15 19:57:48,507 - WARNING - at 10km height this is a difference of about 4 indices
2026-09-15 19:57:48,509 - INFO - Starting Overlap retrieval
2026-09-15 19:57:48,510 - INFO - Cloud free segment: 0. Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-09-15 19:57:48,511 - INFO - Channels: 355 total FR | 355 total NR.
2026-09-15 19:57:48,796 - INFO - Channels: 387 total FR | 387 total NR.
2026-09-15 19:57:49,098 - INFO - Channels: 532 total FR | 532 total NR.
2026-09-15 19:57:49,403 - INFO - Channels: 607 total FR | 607 total NR.
2026-09-15 19:57:49,698 - INFO - Cloud free segment: 1. Time: 2024-08-21T01:12:00.000000 - 2024-08-21T02:02:00.000000.
2026-09-15 19:57:49,699 - INFO - Channels: 355 total FR | 355 total NR.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\

In [28]:
## Aggregate overlap corrected profiles
data_cube.aggregate_profiles(var=['sigOLCor', 'BGOLCor'])

2026-09-15 19:57:56,761 - INFO - Aggregating variable: sigOLCor ...
2026-09-15 19:57:56,834 - INFO - Aggregating variable: BGOLCor ...


In [29]:
## Klett retrieval for overlap corrected profiles
data_cube.retrievalKlett(oc=True)

## Raman retrieval for overlap corrected profiles
data_cube.retrievalRaman(oc=True)

2026-09-15 19:57:56,853 - INFO - Klett retrieval for FR None overlap corrected signal ...
2026-09-15 19:57:56,855 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-09-15 19:57:56,861 - WARNING - at 10km height this is a difference of about 4 indices
2026-09-15 19:57:56,863 - INFO - Cloud free segment 0, Time: 2024-08-21T00:00:00.000000 - 2024-08-21T00:56:30.000000.
2026-09-15 19:57:56,865 - INFO - Channel: 532, total, FR klett.
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:276: RuntimeWarning: invalid value encountered in scalar divide
  denominator1 = RCS[iAlt + 1] / (aerBsc[iAlt + 1] + molBsc[iAlt + 1])
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:310: RuntimeWarning: divide by zero encountered in divide
  aerRelBRStd = np.abs((1 + noise / signal) / (1 + aerBsc + molBsc / 1e3) - 1)
2026-09-15 19:57:56,933 - INFO - Channel: 355, total, FR klett.
2026-09-15 19:57

## Depol and Ångström Profiles

Retrieval of Volume and Particle depolarization as well as Ångström 355/532 backscatter 532/1064 backscatter, and 355/532 Extinction

In [30]:
## Volume and particle depolarization
data_cube.calcDepol()

2026-09-15 19:58:00,939 - INFO - Calculate volume and particle depolarization ratios for product klett ...
2026-09-15 19:58:00,942 - INFO - voldepol at channel 532 cldFree 0 (np.int64(0), np.int64(114))
2026-09-15 19:58:00,944 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.691980299458336 error [ 0.00027  0.02024 -0.00463] Window 25 
2026-09-15 19:58:00,947 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.691980299458336 error [ 0.00027  0.02024 -0.00463] Window 1 
2026-09-15 19:58:00,949 - INFO - voldepol at channel 355 cldFree 0 (np.int64(0), np.int64(114))
2026-09-15 19:58:00,950 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.55786879900888 error [ 0.00016  0.01567 -0.00958] Window 25 
2026-09-15 19:58:00,952 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.55786879900888 error [ 0.00016  0.01567 -0.00958] Window 1 
2026-09-15 19:58:00,953 - INFO - voldepol at channel 1064 cldFree 0 (np.int64(0), np.int64(114))
2026-09-15 19:58:00,955 - INFO - G [1.] [1.] H [-0.02439] [-0.99

In [31]:
## Ångström ratios
data_cube.Angstroem()

2026-09-15 19:58:01,118 - INFO - Calculate Angstrom exponents for product klett ...
2026-09-15 19:58:01,118 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Bsc.
2026-09-15 19:58:01,121 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Bsc.
2026-09-15 19:58:01,122 - INFO - Channels: 532_total_FR, 1064_total_FR. Product: Bsc.
2026-09-15 19:58:01,124 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Ext.
2026-09-15 19:58:01,126 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Ext.
2026-09-15 19:58:01,127 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Bsc.
2026-09-15 19:58:01,129 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Bsc.
2026-09-15 19:58:01,130 - INFO - Channels: 355_total_FR, 532_total_FR. Product: Ext.
2026-09-15 19:58:01,131 - INFO - Channels: 355_total_NR, 532_total_NR. Product: Ext.
2026-09-15 19:58:01,132 - INFO - Calculate Angstrom exponents for product raman ...
2026-09-15 19:58:01,133 - INFO - Channels: 355_total_FR, 532_total

## Lidar Calibration

- Lidar calibration constants (LC) are retrieved for each channel at each cloud free period for both Klett and Raman retrieved profiles.
- If enabled, from the time period [24h before the measurement, 24 after the measurement] are loaded from a dedicated database.
- The optimal LCs. ie. the ones used for processing, are chosen from the set of retrieved and loaded LCs based on their respective uncertainty. Prioritizing LCs from Raman retrieved profiles over those from Klett retrieved profiles.
- If no LCs are found, default values will be used.

In [32]:
## Lidar calibration for both Klett and Raman retrieval
data_cube.LidarCalibration()

2026-09-15 19:58:01,167 - INFO - Performing Lidar calibration ...
2026-09-15 19:58:01,168 - INFO - LC retrieval: klett method
2026-09-15 19:58:01,176 - INFO - Using Retrieved Extinction
2026-09-15 19:58:01,235 - INFO - cldFreGrp 0, Channel 532 total FR, LC_stable 110474202028031.84, LCStd 0.0022342997417183915
2026-09-15 19:58:01,243 - INFO - Using Retrieved Extinction
2026-09-15 19:58:01,303 - INFO - cldFreGrp 0, Channel 355 total FR, LC_stable 25663840488250.824, LCStd 0.001697895966145093
2026-09-15 19:58:01,311 - INFO - Using Retrieved Extinction
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\helper.py:1256: RuntimeWarning: Mean of empty slice
  thisMean = np.nanmean(window)
2026-09-15 19:58:01,374 - INFO - cldFreGrp 0, Channel 1064 total FR, LC_stable 72795106684802.06, LCStd 0.002999019147608419
2026-09-15 19:58:01,381 - INFO - Using Retrieved Extinction
2026-09-15 19:58:01,446 - INFO - cldFreGrp 0, Channel 532 total NR, LC_stable 7337054355591.706, LCStd 0.001418580875

In [33]:
## Display Lidar calibration constants per channel
data_cube.LCused

{'355_total_FR': {'LC': np.float64(17022111394748.135),
  'LCStd': np.float64(62244693763.10861),
  'method': np.str_('raman')},
 '532_total_FR': {'LC': np.float64(113784427765301.78),
  'LCStd': np.float64(431405182550.1493),
  'method': np.str_('raman')},
 '1064_total_FR': {'LC': np.float64(56376092629819.53),
  'LCStd': np.float64(477066474909.2084),
  'method': np.str_('raman')},
 '387_total_FR': {'LC': np.float64(48925494588027.48),
  'LCStd': np.float64(37183616636.410286),
  'method': np.str_('raman')},
 '607_total_FR': {'LC': np.float64(290870505264845.75),
  'LCStd': np.float64(235197470459.66364),
  'method': np.str_('raman')},
 '355_total_NR': {'LC': np.float64(2755544929966.0894),
  'LCStd': np.float64(10940832565.725788),
  'method': np.str_('raman')},
 '532_total_NR': {'LC': np.float64(8844189457722.008),
  'LCStd': np.float64(40832256989.268745),
  'method': np.str_('raman')},
 '387_total_NR': {'LC': np.float64(4006008627669.4106),
  'LCStd': np.float64(5828166146.213741

In [34]:
## Store calibration constants in database
data_cube.write_2_sql_db(parameter='DC')
data_cube.write_2_sql_db(parameter='WVC', method='model_profile')
data_cube.write_2_sql_db(parameter='WVC', method='model_regression')
data_cube.write_2_sql_db(parameter='LC', method='raman')
data_cube.write_2_sql_db(parameter='LC', method='klett')

2026-09-15 19:58:03,114 - INFO - read db_path from polly_config_dict pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,114 - INFO - writing to sqlite-db: pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,115 - INFO - writing DC to table: depol_calibration_constant
2026-09-15 19:58:03,121 - INFO - 5 rows inserted into 'depol_calibration_constant'.
2026-09-15 19:58:03,122 - INFO - read db_path from polly_config_dict pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,123 - INFO - writing to sqlite-db: pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,124 - INFO - writing WVC to table: wv_calibration_constant
2026-09-15 19:58:03,129 - INFO - 2 rows inserted into 'wv_calibration_constant'.
2026-09-15 19:58:03,130 - INFO - read db_path from polly_config_dict pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,131 - INFO - writing to sqlite-db: pollyxt_cpv\pollyxt_cpv_calibration_v3.db
2026-09-15 19:58:03,131 - INFO - writing WVC to tab

In [35]:
## Save retrieved optical profiles
write_profile2nc_file(data_cube, prod_ls=["profiles", "NR_profiles", "OC_profiles"])

2026-09-15 19:58:03,166 - INFO - saving product: profiles
2026-09-15 19:58:03,273 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-0056_profiles.nc
2026-09-15 19:58:03,492 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0112-0202_profiles.nc
2026-09-15 19:58:03,599 - INFO - saving product: NR_profiles
2026-09-15 19:58:03,705 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-0056_NR_profiles.nc
2026-09-15 19:58:03,946 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0112-0202_NR_profiles.nc
2026-09-15 19:58:03,998 - INFO - saving product: OC_profiles
2026-09-15 19:58:04,109 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-0056_OC_profiles.nc
2026-09-15 19:58:04,332 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0112-0202_OC_profiles.nc


## High Resoulution Retrievals

The following high resolution (30s) time-height data are retrieved: 

- Attenuated backscatter
- Volume depolarization
- Molecular backscatter and extinction
- Quality mask
- QuasiV1 and QuasiV2 retrievals
- Target categorization V1 and V2

In [36]:
## Highres attenuated backscatter and volume depolarization
data_cube.attBsc_volDepol()

## Highres molecular signal
data_cube.molecularHighres()

2026-09-15 19:58:04,452 - INFO - 2D attenuated backscatter retrieval ...


2026-09-15 19:58:05,421 - WARNING - Exprimental, attenuated backscatter solution for 387_total_NR
2026-09-15 19:58:05,621 - INFO - Channel: 355 total FR | 355 cross FR
2026-09-15 19:58:05,817 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 47.55786879900888 error [ 0.00016  0.01567 -0.00958] Window 1 
2026-09-15 19:58:06,347 - INFO - Channel: 532 total FR | 532 cross FR
2026-09-15 19:58:06,542 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.691980299458336 error [ 0.00027  0.02024 -0.00463] Window 1 
2026-09-15 19:58:07,067 - INFO - Channel: 1064 total FR | 1064 cross FR
2026-09-15 19:58:07,311 - INFO - G [1.] [1.] H [-0.02439] [-0.996] Eta 0.18186339128701293 error [ 0.00133  0.02379 -0.01205] Window 1 
2026-09-15 19:58:08,276 - INFO - 2D volume depolarization ratio retrieval ...
2026-09-15 19:58:08,415 - INFO - G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.691980299458336 error [ 0.00027  0.02024 -0.00463] Window 1 
2026-09-15 19:58:08,889 - INFO - G [1.] [1.] H [0.02041] [-0.998] Eta 4

In [37]:
## Quality mask of signal
data_cube.estQualityMask()

2026-09-15 19:58:11,948 - INFO - Estimate quality masks ...
2026-09-15 19:58:11,949 - INFO - Calculating SNR from smoothed signal to improve data quality...
2026-09-15 19:58:12,392 - INFO - Vectorized group 0, Nr = 3, Nc = 10 ...
2026-09-15 19:58:29,208 - INFO - Using improved SNR in quality mask estimation.


In [38]:
## Highres water vapor mixing ratio
data_cube.wvmr()

2026-09-15 19:58:29,778 - INFO - 2D water vapor mixing ratio retrieval ...
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\highres.py:199: RuntimeWarning: divide by zero encountered in divide
  wvmr_raw = (sig407 / sig387) * (trans_387 / trans_407)


In [39]:
## QuasiV1 retrievals and Target categorization
data_cube.quasiV1()

2026-09-15 19:58:37,252 - INFO - Calculating Quasi V1 particle backscatter coefficient
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:134: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att)**2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:134: RuntimeWarning: overflow encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att)**2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:136: RuntimeWarning: overflow encountered in multiply
  quasi_par_ext = quasi_par_bsc * LRaer
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:133: RuntimeWarning: overflow encountered in multiply
  quasi_par_att = np.exp(-np.nancumsum(quasi_par_ext * diff_height, axis=1))
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: overflow encountered in ac

In [40]:
## QuasiV2 retrievals and Target categorization
data_cube.quasiV2()

2026-09-15 19:59:03,212 - INFO - Calculating Quasi V2 particle backscatter coefficient
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:170: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = (att_beta_el / att_beta_ra) * quasi_par_att - molBscEl
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:169: RuntimeWarning: overflow encountered in exp
  quasi_par_att = np.exp((1 - (wv / wv_r)**AE) * OD_par + (OD_mol - OD_mol_r)) * molBscEl
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:170: RuntimeWarning: invalid value encountered in multiply
  quasi_par_bsc = (att_beta_el / att_beta_ra) * quasi_par_att - molBscEl
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:167: RuntimeWarning: ov

In [41]:
## Save highres retrievals
write2nc_file(data_cube, prod_ls=["att_bsc", "NR_att_bsc", "OC_att_bsc", "vol_depol", "quasi_results", "quasi_results_V2", "target_classification", "target_classification_V2"])

2026-09-15 19:59:36,478 - INFO - saving product: att_bsc
2026-09-15 19:59:36,958 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_att_bsc.nc
2026-09-15 19:59:47,301 - INFO - saving product: NR_att_bsc
2026-09-15 19:59:47,678 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_NR_att_bsc.nc
2026-09-15 19:59:53,928 - INFO - saving product: OC_att_bsc
2026-09-15 19:59:54,046 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_OC_att_bsc.nc
2026-09-15 19:59:59,176 - INFO - saving product: vol_depol
2026-09-15 19:59:59,297 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_vol_depol.nc
2026-09-15 20:00:03,581 - INFO - saving product: quasi_results
2026-09-15 20:00:03,697 - INFO - writing to file: pollyxt_cpv\2024\08\21\2024_08_21_pollyxt_cpv_0000-2359_quasi_results.nc
2026-09-15 20:00:08,981 - INFO - saving product: quasi_results_V2
2026-09-15 20:00:09,095 - INFO - wri